In [1]:
!pip install peft transformers bitsandbytes accelerate -q

In [2]:
from google.colab import files

adapter_file=files.upload()

Saving benchmark_report.json to benchmark_report.json
Saving dataset_test.json to dataset_test.json
Saving qwen-lora-adapter.zip to qwen-lora-adapter.zip


In [3]:
import shutil

In [4]:
shutil.unpack_archive("qwen-lora-adapter.zip","qwen-lora-adapter")

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name="Qwen/Qwen2.5-1.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [6]:
import torch

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
    )

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
from peft import PeftModel

model=PeftModel.from_pretrained(model, "./qwen-lora-adapter")

In [8]:
import json

with open("dataset_test.json","r") as f:
 test_data=json.load(f)

In [9]:
system_prompt = """You are a product data extraction assistant. Extract structured information from the given product description and return a valid JSON object with these fields:
- product_name: string
- brand: string or null if unknown
- category: string
- price: {"amount": number, "currency": string} or null if not mentioned
- key_specs: object with relevant specifications
- condition: "new", "used", or "refurbished"

Return ONLY the JSON object, no other text."""

def build_prompt(product_description):
  messages=[
      {'role':'system', 'content':system_prompt},
      {'role':'user', 'content':f"Extract the product data from this description:\n\n{product_description}"}
  ]

  prompt=tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  return prompt

def generate_response(prompt):
   inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

   with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
      )

   new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
   response = tokenizer.decode(new_tokens, skip_special_tokens=True)
   return response

results=[]

for i,eachData in enumerate(test_data):
  prompt=build_prompt(eachData["input"])
  response=generate_response(prompt)

  results.append({
      "input":eachData["input"],
      "expected":eachData["output"],
      "predicted":response
  })

  if (i + 1) % 10 == 0:
        print(f"Completed {i + 1}/80")

print("All 80 examples done!")

Completed 10/80
Completed 20/80
Completed 30/80
Completed 40/80
Completed 50/80
Completed 60/80
Completed 70/80
Completed 80/80
All 80 examples done!


In [10]:
import re

def parse_json_response(response_text):
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass

    code_block = re.search(r"```(?:json)?\s*(.*?)\s*```", response_text, re.DOTALL)
    if code_block:
        try:
            return json.loads(code_block.group(1))
        except json.JSONDecodeError:
            pass

    brace_match = re.search(r"\{.*\}", response_text, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group())
        except json.JSONDecodeError:
            pass

    return None

In [11]:
def normalize(value):
    if value is None:
        return None
    if isinstance(value, str):
        return value.strip().lower()
    return value

def compare_price(predicted, expected):
    if predicted is None and expected is None:
        return True
    if predicted is None or expected is None:
        return False
    if not isinstance(predicted, dict) or not isinstance(expected, dict):
        return False
    return predicted.get("amount") == expected.get("amount")

def score_example(predicted_dict, expected_dict):
    scores = {}
    scores["product_name"] = normalize(predicted_dict.get("product_name")) == normalize(expected_dict.get("product_name"))
    scores["brand"] = normalize(predicted_dict.get("brand")) == normalize(expected_dict.get("brand"))
    scores["category"] = normalize(predicted_dict.get("category")) == normalize(expected_dict.get("category"))
    scores["price"] = compare_price(predicted_dict.get("price"), expected_dict.get("price"))
    scores["condition"] = normalize(predicted_dict.get("condition")) == normalize(expected_dict.get("condition"))
    return scores

In [12]:
valid_json_count = 0
field_scores = {"product_name": 0, "brand": 0, "category": 0, "price": 0, "condition": 0}
failed_parses = []

for i, result in enumerate(results):
    predicted_dict = parse_json_response(result["predicted"])
    expected_dict = result["expected"]
    if isinstance(expected_dict, str):
        expected_dict = json.loads(expected_dict)
    if predicted_dict is None:
        failed_parses.append(i)
        continue
    valid_json_count += 1
    scores = score_example(predicted_dict, expected_dict)
    for field, correct in scores.items():
        if correct:
            field_scores[field] += 1

print(f"Valid JSON: {valid_json_count}/80 ({valid_json_count/80*100:.1f}%)")
print(f"\nField-level accuracy (out of {valid_json_count} valid responses):")
print("-" * 45)
for field, count in field_scores.items():
    pct = count / valid_json_count * 100 if valid_json_count > 0 else 0
    print(f"  {field:15s}: {count}/{valid_json_count} ({pct:.1f}%)")
print(f"\nFailed to parse: {len(failed_parses)} examples")
if failed_parses:
    print(f"Failed indices: {failed_parses}")

Valid JSON: 80/80 (100.0%)

Field-level accuracy (out of 80 valid responses):
---------------------------------------------
  product_name   : 51/80 (63.7%)
  brand          : 73/80 (91.2%)
  category       : 61/80 (76.2%)
  price          : 79/80 (98.8%)
  condition      : 73/80 (91.2%)

Failed to parse: 0 examples


In [13]:
with open("benchmark_report.json", "r") as f:
    baseline = json.load(f)

print("=" * 60)
print("COMPARISON: Base Model vs Fine-Tuned Model")
print("=" * 60)
print(f"\n{'Metric':<20} {'Base Model':>12} {'Fine-Tuned':>12} {'Change':>10}")
print("-" * 60)

# Valid JSON comparison
base_json = baseline["valid_json_rate"]
ft_json = round(valid_json_count / 80 * 100, 1)
print(f"{'Valid JSON %':<20} {base_json:>11.1f}% {ft_json:>11.1f}% {ft_json - base_json:>+9.1f}%")

# Field-level comparison
for field in ["product_name", "brand", "category", "price", "condition"]:
    base_pct = baseline["field_accuracy"][field]["percentage"]
    ft_pct = round(field_scores[field] / valid_json_count * 100, 1) if valid_json_count > 0 else 0
    print(f"{field:<20} {base_pct:>11.1f}% {ft_pct:>11.1f}% {ft_pct - base_pct:>+9.1f}%")

print("=" * 60)

COMPARISON: Base Model vs Fine-Tuned Model

Metric                 Base Model   Fine-Tuned     Change
------------------------------------------------------------
Valid JSON %                97.5%       100.0%      +2.5%
product_name                46.2%        63.7%     +17.5%
brand                       88.5%        91.2%      +2.7%
category                    39.7%        76.2%     +36.5%
price                       96.2%        98.8%      +2.6%
condition                   87.2%        91.2%      +4.0%


In [14]:
comparison_report = {
    "base_model": baseline,
    "fine_tuned": {
        "model": "Qwen2.5-1.5B-Instruct + LoRA",
        "total_examples": 80,
        "valid_json": valid_json_count,
        "valid_json_rate": round(valid_json_count / 80 * 100, 1),
        "field_accuracy": {}
    }
}

for field, count in field_scores.items():
    pct = round(count / valid_json_count * 100, 1) if valid_json_count > 0 else 0
    comparison_report["fine_tuned"]["field_accuracy"][field] = {
        "correct": count,
        "total": valid_json_count,
        "percentage": pct
    }

with open("comparison_report.json", "w") as f:
    json.dump(comparison_report, f, indent=2)

with open("finetuned_predictions.json", "w") as f:
    json.dump(results, f, indent=2)

from google.colab import files
files.download("comparison_report.json")
files.download("finetuned_predictions.json")

print("Reports saved and downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Reports saved and downloaded!
